# Neo4j MCP Agent with Strands Agents

Query a Neo4j graph database using **AWS Strands Agents** and **AgentCore Gateway MCP**.



## 1. Setup

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client

print("Imports OK")

## 2. Configuration


In [ ]:
#################################################
# CONFIGURATION
# 1. Set MODEL to match what you used with setup-inference-profile.sh
# 2. Paste your inference profile ARN
#################################################

MODEL = "haiku"  # Options: haiku, sonnet, sonnet4, sonnet45
INFERENCE_PROFILE_ARN = "PASTE_YOUR_ARN_HERE"  # <-- PASTE HERE
REGION = "us-west-2"

# MCP Gateway credentials (from .mcp-credentials.json)
GATEWAY_URL = "PASTE_YOUR_GATEWAY_URL_HERE"
ACCESS_TOKEN = "PASTE_YOUR_ACCESS_TOKEN_HERE"

#################################################

# Model ID mapping (for reference/validation only - Strands uses ARN directly)
BASE_MODEL_IDS = {
    "haiku": "anthropic.claude-3-5-haiku-20241022-v1:0",
    "sonnet": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    "sonnet4": "anthropic.claude-sonnet-4-20250514-v1:0",
    "sonnet45": "anthropic.claude-sonnet-4-5-20250929-v1:0",
}

# Validate configuration
errors = []
if "PASTE" in INFERENCE_PROFILE_ARN or "YOUR" in INFERENCE_PROFILE_ARN:
    errors.append("Paste your inference profile ARN (run ./setup-inference-profile.sh)")
if MODEL not in BASE_MODEL_IDS:
    errors.append(f"Unknown MODEL '{MODEL}'. Valid: {list(BASE_MODEL_IDS.keys())}")
if "PASTE" in GATEWAY_URL or "PASTE" in ACCESS_TOKEN:
    errors.append("Replace GATEWAY_URL and ACCESS_TOKEN (from .mcp-credentials.json)")

if errors:
    print("ERROR: Configuration incomplete!")
    for e in errors:
        print(f"  - {e}")
else:
    BASE_MODEL_ID = BASE_MODEL_IDS[MODEL]
    print(f"Model:   {MODEL} ({BASE_MODEL_ID})")
    print(f"Profile: {INFERENCE_PROFILE_ARN}")
    print(f"Region:  {REGION}")
    print(f"Gateway: {GATEWAY_URL[:50]}...")
    print(f"Token:   {ACCESS_TOKEN[:30]}...")
    print("\nConfiguration OK!")

## 3. Initialize Model & MCP Client

**Key pattern from AWS sample:** The transport factory returns a fresh `streamablehttp_client` each time, with the Bearer token embedded in headers.

In [ ]:
# Bedrock model - uses inference profile ARN directly
# Note: Strands BedrockModel doesn't have base_model_id parameter like LangChain
model = BedrockModel(
    model_id=INFERENCE_PROFILE_ARN,
    region_name=REGION,
    temperature=0,
)


# Token getter (called each time transport is created)
def get_token():
    return ACCESS_TOKEN


# Transport factory - returns fresh streamablehttp_client each call
def create_streamable_http_transport():
    return streamablehttp_client(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {get_token()}"}
    )


# MCP client with transport factory
mcp_client = MCPClient(create_streamable_http_transport)

print(f"Model initialized: {MODEL} ({BASE_MODEL_ID})")
print("MCP client ready")

## 4. Test MCP Connection

In [ ]:
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Connected! Found {len(tools)} tools:")
    for tool in tools:
        print(f"  - {tool.tool_spec['name']}")

## 5. Create Agent & Query Function

In [ ]:
SYSTEM_PROMPT = """You are a Neo4j database assistant. You can:
- Get the database schema
- Run read-only Cypher queries

Always get the schema first, then query based on actual labels/relationships.
Be concise. Format results clearly."""


def query(question: str) -> str:
    """Query the Neo4j database via MCP."""
    print(f"Q: {question}")
    print("-" * 60)
    
    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(
            model=model,
            tools=tools,
            system_prompt=SYSTEM_PROMPT,
        )
        result = agent(question)
    
    print(f"\nA: {result}")
    return str(result)

## 6. Demo Queries

In [ ]:
_ = query("What is the database schema?")

In [ ]:
_ = query("How many nodes are there by label?")

In [ ]:
_ = query("Show 5 sample records from the most populated node type.")

## 7. Your Query

In [ ]:
# _ = query("Your question here")